# FE Vector Embedding v1

Notebook Kaggle này đọc keyframe/frame đã trích xuất trên Google Cloud Storage,
chạy OpenCLIP image embedding, rồi xuất artifact phục vụ hai nhánh import:

- PostgreSQL/Supabase: `datasets`, `videos`, `shots`, `keyframes`, `events`, `event_keyframes`.
- Zilliz/Milvus: `keyframe_embeddings` và tùy chọn `event_embeddings`.

Bố cục bám theo format `Config`, `Data`, `Model`, `Run`. Mỗi lần chạy tạo một
thư mục run riêng trong `/kaggle/working/feature_extraction_runs/<run_id>/`.

## How To Run

1. Bật GPU cho Kaggle Notebook.
2. Bật Internet nếu cần cài package hoặc tải OpenCLIP weights.
3. Thêm Kaggle Secrets `GCS_BUCKET` và `GCS_CREDENTIALS_JSON`.
4. Sửa cell `Config` cho batch, prefix, batch size, workers và run mode.
5. Chạy `Dry Run`, sau đó `Smoke Test`, rồi `Demo 1 Batch`.
6. Khi output đúng, đặt `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"` ở cell full run.

# Install Dependencies

Cell này cài dependency tối thiểu cho Kaggle: GCS client, OpenCLIP, PyTorch utilities,
pandas/numpy và progress bar. Nếu Kaggle đã có sẵn package, cell sẽ chạy rất nhanh.

In [ ]:
!pip install -q google-cloud-storage google-auth open_clip_torch tqdm pandas numpy pillow


# Config

Chỉ sửa cell này khi đổi bucket, batch, GCS prefix, model, batch size, worker, giới hạn
dry/smoke/demo/full, hoặc bật/tắt upload artifact. Không hard-code service account JSON
nếu notebook có thể được chia sẻ.

In [ ]:
from pathlib import Path

# GCS and dataset identity
GCS_BUCKET = ""  # Empty means read Kaggle Secret/ENV named GCS_BUCKET.
GCS_CREDENTIALS_JSON = ""  # Empty means read Kaggle Secret/ENV below.
GCS_CREDENTIALS_FILE = ""  # Optional local JSON file path.
GCS_BUCKET_SECRET_NAME = "GCS_BUCKET"
GCS_CREDENTIALS_JSON_SECRET_NAMES = ["GCS_CREDENTIALS_JSON", "GCS_SERVICE_ACCOUNT_JSON"]
GCS_TIMEOUT_SECONDS = 60

DATASET_ID = "ai_challenge_2025"
DATASET_CODE = "aic-2026"
DATASET_NAME = "aic-ai-challenge-2025"
DATASET_VERSION = "v1"
DATASET_DB_ID = ""  # Empty means deterministic uuid5(DATASET_CODE, DATASET_VERSION).
SOURCE_VERSION = "kaggle_current"
PROFILE_VERSION = "autoshot_v1"

EXPECTED_BATCHES = [f"L{i:02d}" for i in range(21, 31)]
BATCHES = "L21"  # "all", "L21,L22", or ["L21", "L22"].
VIDEO_IDS = []  # Optional allow-list, e.g. ["L21_V001", "L21_V002"].
MAX_VIDEOS = None
MAX_FRAMES = None

# Frame extraction output from scripts/loaders/README.MD.
KEYFRAMES_PREFIX = "processed/keyframes"
MANIFESTS_PREFIX = "processed/keyframes_manifests"
FRAME_PREFIX_TEMPLATE = (
    "{keyframes_prefix}/dataset={dataset_id}/batch={batch_id}/"
    "profile={profile_version}/"
)
MANIFEST_PREFIX_TEMPLATE = (
    "{manifests_prefix}/dataset={dataset_id}/batch={batch_id}/"
    "profile={profile_version}/"
)

# Preferred: map batch -> frame-extraction run_id or explicit shot_segments.csv URI.
# If both are empty and AUTO_DISCOVER_SHOT_SEGMENTS=True, notebook picks latest
# *_SUCCESS run under processed/keyframes_manifests/... for each batch.
KEYFRAME_RUN_ID_BY_BATCH = {
    # "L21": "full_autoshot_l21_20260718_120000",
}
SHOT_SEGMENTS_URI_BY_BATCH = {
    # "L21": "gs://aic_ai_2026/processed/keyframes_manifests/dataset=ai_challenge_2025/batch=L21/profile=autoshot_v1/run_id=<run_id>/shot_segments.csv",
}
AUTO_DISCOVER_SHOT_SEGMENTS = True
REQUIRE_SUCCESS_FOR_DISCOVERY = True
USE_SHOT_SEGMENTS_METADATA = True
DEFAULT_FPS = 25.0

# Output layout
TASK_NAME = "fe_vector_embedding_v1"
FEATURE_DIR_NAME = "vit-ViT-B-32-laion2b_s34b_b79k"
EMBEDDING_MODEL_VERSION = FEATURE_DIR_NAME
OUTPUT_PREFIX = "processed/features/fe-vector-embedding-v1"
LOCAL_RUN_ROOT = Path("/kaggle/working/feature_extraction_runs")
RUN_ID = ""  # Empty means auto-generate per mode.

UPLOAD_ARTIFACTS_TO_GCS = True
UPLOAD_WORKERS = 8
WRITE_ZILLIZ_JSONL = True
ZILLIZ_KEYFRAME_COLLECTION = "keyframe_embeddings"
ZILLIZ_EVENT_COLLECTION = "event_embeddings"
ZILLIZ_SHARD_SIZE = 4096
VECTOR_ROUND_DECIMALS = 6

# Model and performance tuning
DEVICE = "auto"  # auto, cuda, cuda:0, cpu.
OPENCLIP_MODEL_NAME = "ViT-B-32"
OPENCLIP_PRETRAINED = "laion2b_s34b_b79k"
MODEL_CACHE_DIR = Path("/kaggle/working/model-cache")
EMBED_BATCH_SIZE = 256
PIPELINE_BATCH_SIZE = 512
DOWNLOAD_WORKERS = 32
IMAGE_DECODE_WORKERS = 2
DATALOADER_PREFETCH_FACTOR = 2
PIN_MEMORY = True
PERSISTENT_WORKERS = True
EMBED_PRECISION = "fp16"  # fp16, bf16, fp32.
L2_NORMALIZE = True
ALLOW_TF32 = True
CUDNN_BENCHMARK = True
TORCH_COMPILE = False
DELETE_LOCAL_AFTER_BATCH = True
REUSE_LOCAL_DOWNLOADS = True
CONTINUE_ON_BATCH_ERROR = True


# Event embedding, based on event-embeddings.ipynb
BUILD_EVENTS = True
MAX_TIME_GAP_SEC = 6.0
SCENE_SIMILARITY_THRESHOLD = 0.72
MAX_EVENT_DURATION_SEC = 45.0  # None disables the cap.
SEGMENTATION_VERSION = "event-openclip-greedy-v1"

# Run presets
DRY_RUN_MAX_FRAMES = 20
SMOKE_TEST_BATCHES = "L21"
SMOKE_TEST_MAX_FRAMES = 16
SMOKE_TEST_UPLOAD_ARTIFACTS = False

DEMO_BATCHES = "L21"
DEMO_BATCH_SIZE = 64
DEMO_BATCH_INDEX = 0
DEMO_UPLOAD_ARTIFACTS = True

FULL_BATCHES = "all"
FULL_MAX_VIDEOS = None
FULL_MAX_FRAMES = None
CONFIRM_FULL_RUN = ""  # Set to "RUN_FULL_DATASET" before full run.

print("Config ready")
print("BATCHES:", BATCHES)
print("FEATURE_DIR_NAME:", FEATURE_DIR_NAME)
print("PIPELINE_BATCH_SIZE:", PIPELINE_BATCH_SIZE)
print("EMBED_BATCH_SIZE:", EMBED_BATCH_SIZE)
print("DOWNLOAD_WORKERS:", DOWNLOAD_WORKERS)
print("IMAGE_DECODE_WORKERS:", IMAGE_DECODE_WORKERS)


# Runtime Setup

Cell này import thư viện, cấu hình logging, đọc Kaggle Secrets an toàn, tạo helper
thời gian/run id, và chuẩn hóa bucket/prefix GCS.

In [ ]:
from __future__ import annotations

import csv
import json
import logging
import math
import os
import re
import shutil
import time
import uuid
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path, PurePosixPath
from typing import Any, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

FRAME_IDX_RE = re.compile(r"f(\d+)", re.IGNORECASE)
SHOT_INDEX_RE = re.compile(r"(?:shot[_-]?|_S)(\d+)", re.IGNORECASE)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
RUN_STARTED_AT = time.perf_counter()


def utc_now() -> str:
    """Return the current UTC timestamp as an ISO-8601 string."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def millis_since(started: float) -> int:
    """Return elapsed milliseconds since a perf_counter timestamp."""
    return int((time.perf_counter() - started) * 1000)


def normalize_prefix(value: str) -> str:
    """Normalize a GCS object prefix without leading or trailing slashes."""
    return str(value or "").replace("\\", "/").strip("/")


def read_kaggle_secret(secret_name: str, default: str = "") -> str:
    """Read a Kaggle Secret, falling back to an environment variable."""
    if not secret_name:
        return default
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(secret_name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(secret_name, default)


def read_first_secret(secret_names: list[str] | tuple[str, ...]) -> str:
    """Return the first non-empty Kaggle Secret or environment variable."""
    for name in secret_names:
        value = read_kaggle_secret(str(name), "")
        if value:
            return value
    return ""


def parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Split a gs://bucket/path URI into bucket name and blob path."""
    if not uri.startswith("gs://"):
        raise ValueError(f"Expected a gs:// URI, got {uri!r}")
    rest = uri[len("gs://") :]
    bucket, blob = rest.split("/", 1)
    return bucket, blob


def resolve_gcs_bucket() -> str:
    """Resolve the effective GCS bucket from config, Kaggle Secret, or env."""
    raw = (GCS_BUCKET or read_kaggle_secret(GCS_BUCKET_SECRET_NAME, "")).strip()
    if raw.startswith("gs://"):
        bucket, _ = parse_gcs_uri(raw.rstrip("/") + "/_")
        return bucket
    if not raw:
        raise ValueError("Missing GCS bucket. Set GCS_BUCKET or Kaggle Secret GCS_BUCKET.")
    return raw.strip("/")


def resolve_dataset_db_id() -> str:
    """Return configured dataset UUID or a deterministic uuid5 value."""
    if DATASET_DB_ID:
        return DATASET_DB_ID
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"dataset:{DATASET_CODE}:{DATASET_VERSION}"))


def make_run_id(mode: str) -> str:
    """Build a stable run id for local folders and uploaded artifacts."""
    if RUN_ID:
        return RUN_ID
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return f"{TASK_NAME}-{mode}-{stamp}-{uuid.uuid4().hex[:8]}"


def make_storage_client():
    """Create a Google Cloud Storage client from Kaggle Secrets or default auth."""
    from google.cloud import storage
    from google.oauth2 import service_account

    credentials_json = (GCS_CREDENTIALS_JSON or read_first_secret(GCS_CREDENTIALS_JSON_SECRET_NAMES)).strip()
    credentials_file = (GCS_CREDENTIALS_FILE or os.getenv("GCS_CREDENTIALS_FILE", "")).strip()

    if credentials_json:
        if credentials_json.startswith("{"):
            info = json.loads(credentials_json)
            credentials = service_account.Credentials.from_service_account_info(info)
            return storage.Client(project=credentials.project_id, credentials=credentials)
        if Path(credentials_json).exists():
            return storage.Client.from_service_account_json(credentials_json)

    if credentials_file:
        return storage.Client.from_service_account_json(credentials_file)

    return storage.Client()


def setup_console_logger(name: str, log_path: Path | None = None) -> logging.Logger:
    """Create a console/file logger that is friendly inside Kaggle notebooks."""
    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        file_handler = logging.FileHandler(log_path, encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
    return logger


LOGGER = setup_console_logger(TASK_NAME)
print("Runtime ready at", utc_now())


# Data

Cell này định nghĩa schema `FrameItem`, đọc `shot_segments.csv` khi có, fallback sang
list ảnh trực tiếp dưới `processed/keyframes`, và tạo inventory frame theo batch/video.

In [ ]:
@dataclass(frozen=True)
class FrameItem:
    """Metadata for one extracted keyframe image stored in GCS."""

    dataset_id: str
    batch_id: str
    video_id: str
    video_name: str
    video_gcs_uri: str
    shot_id: str
    shot_index: int
    shot_start_frame: int
    shot_end_frame: int
    shot_start_sec: float
    shot_end_sec: float
    frame_type: str
    frame_idx: int
    frame_seconds: float
    keyframe_id: str
    image_rel_path: str
    image_gcs_uri: str
    image_storage_key: str
    fps: float | None = None
    total_frames: int | None = None
    source: str = "gcs"

    @property
    def image_name(self) -> str:
        """Return the image filename without parent folders."""
        return PurePosixPath(self.image_storage_key).name or PurePosixPath(self.image_rel_path).name

    @property
    def timestamp_ms(self) -> int:
        """Return the frame timestamp in milliseconds."""
        return int(float(self.frame_seconds) * 1000)


def bool_value(raw: object, default: bool = False) -> bool:
    """Convert common CSV truthy/falsey values to bool."""
    if raw is None or raw == "":
        return default
    if isinstance(raw, bool):
        return raw
    return str(raw).strip().lower() in {"1", "true", "t", "yes", "y"}


def int_value(raw: object, default: int = 0) -> int:
    """Convert a nullable CSV value to int."""
    if raw is None or raw == "":
        return default
    return int(float(raw))


def optional_int(raw: object) -> int | None:
    """Convert a nullable CSV value to int or None."""
    if raw is None or raw == "":
        return None
    return int(float(raw))


def float_value(raw: object, default: float = 0.0) -> float:
    """Convert a nullable CSV value to float."""
    if raw is None or raw == "":
        return default
    return float(raw)


def optional_float(raw: object) -> float | None:
    """Convert a nullable CSV value to float or None."""
    if raw is None or raw == "":
        return None
    return float(raw)


def selected_batches(raw_batches: Any) -> list[str]:
    """Normalize batch config into an ordered list of batch ids."""
    if raw_batches in (None, "", "all"):
        return list(EXPECTED_BATCHES)
    if isinstance(raw_batches, str):
        return [item.strip().upper() for item in raw_batches.split(",") if item.strip()]
    return [str(item).strip().upper() for item in raw_batches if str(item).strip()]


def format_prefix_template(template: str, batch_id: str) -> str:
    """Render a GCS prefix template with dataset, batch and profile variables."""
    return normalize_prefix(
        template.format(
            keyframes_prefix=normalize_prefix(KEYFRAMES_PREFIX),
            manifests_prefix=normalize_prefix(MANIFESTS_PREFIX),
            dataset_id=DATASET_ID,
            batch_id=batch_id,
            profile_version=PROFILE_VERSION,
            source_version=SOURCE_VERSION,
        )
    )


def read_text(client: Any, uri_or_path: str, bucket_name: str) -> str:
    """Read UTF-8 text from gs:// URI, bucket-relative path, or local file."""
    if uri_or_path.startswith("gs://"):
        bucket, blob = parse_gcs_uri(uri_or_path)
        return client.bucket(bucket).blob(blob).download_as_text(timeout=GCS_TIMEOUT_SECONDS)
    local_path = Path(uri_or_path)
    if local_path.exists():
        return local_path.read_text(encoding="utf-8-sig")
    return client.bucket(bucket_name).blob(normalize_prefix(uri_or_path)).download_as_text(timeout=GCS_TIMEOUT_SECONDS)


def blob_exists(client: Any, bucket_name: str, blob_name: str) -> bool:
    """Return whether a GCS object exists."""
    return client.bucket(bucket_name).blob(normalize_prefix(blob_name)).exists(timeout=GCS_TIMEOUT_SECONDS)


def discover_latest_shot_segments_uri(client: Any, bucket_name: str, batch_id: str) -> str:
    """Find the latest successful shot_segments.csv for a batch under the manifests prefix."""
    root_prefix = format_prefix_template(MANIFEST_PREFIX_TEMPLATE, batch_id).rstrip("/") + "/"
    blobs = list(client.list_blobs(bucket_name, prefix=root_prefix, timeout=GCS_TIMEOUT_SECONDS))
    success_prefixes = {
        blob.name[: -len("_SUCCESS")]
        for blob in blobs
        if PurePosixPath(blob.name).name == "_SUCCESS"
    }
    candidates = []
    for blob in blobs:
        if PurePosixPath(blob.name).name != "shot_segments.csv":
            continue
        parent_prefix = blob.name[: -len("shot_segments.csv")]
        if REQUIRE_SUCCESS_FOR_DISCOVERY and parent_prefix not in success_prefixes:
            continue
        candidates.append(blob)
    if not candidates:
        return ""
    latest = max(candidates, key=lambda item: item.updated or datetime.min.replace(tzinfo=timezone.utc))
    return f"gs://{bucket_name}/{latest.name}"


def shot_segments_uri_for_batch(client: Any, bucket_name: str, batch_id: str) -> str:
    """Resolve the shot_segments.csv URI for one batch from config or GCS discovery."""
    if batch_id in SHOT_SEGMENTS_URI_BY_BATCH:
        return SHOT_SEGMENTS_URI_BY_BATCH[batch_id]
    if batch_id in KEYFRAME_RUN_ID_BY_BATCH:
        root = format_prefix_template(MANIFEST_PREFIX_TEMPLATE, batch_id).rstrip("/")
        return f"gs://{bucket_name}/{root}/run_id={KEYFRAME_RUN_ID_BY_BATCH[batch_id]}/shot_segments.csv"
    if AUTO_DISCOVER_SHOT_SEGMENTS:
        return discover_latest_shot_segments_uri(client, bucket_name, batch_id)
    return ""


def image_storage_key_from_uri(uri: str) -> str:
    """Return the GCS object key from a gs:// URI."""
    if not uri or not uri.startswith("gs://"):
        return ""
    _, blob = parse_gcs_uri(uri)
    return blob


def parse_frame_idx(image_name: str) -> int | None:
    """Extract frame index from a filename token like f000123."""
    match = FRAME_IDX_RE.search(image_name or "")
    return int(match.group(1)) if match else None


def parse_shot_index(raw_shot_id: object, raw_local: object = None) -> int:
    """Extract zero-based shot index from shot_id_local, shot_index, or shot id text."""
    local = optional_int(raw_local)
    if local is not None:
        return local
    text = str(raw_shot_id or "")
    match = SHOT_INDEX_RE.search(text)
    if match:
        return int(match.group(1))
    return int_value(text, 0)


def normalize_shot_id(video_id: str, raw_shot_id: object, shot_index: int) -> str:
    """Return backend-compatible shot id formatted as <video_id>_S0000."""
    value = str(raw_shot_id or "").strip()
    if value.startswith(video_id + "_S"):
        return value
    return f"{video_id}_S{shot_index:04d}"


def public_url(bucket_name: str, blob_name: str) -> str:
    """Build a stable public HTTPS URL for a GCS object."""
    return f"https://storage.googleapis.com/{bucket_name}/{normalize_prefix(blob_name)}"


def frame_item_from_shot_row(row: dict[str, Any], bucket_name: str, batch_id: str) -> FrameItem | None:
    """Convert one shot_segments.csv row into a FrameItem."""
    if not bool_value(row.get("saved"), True):
        return None
    video_id = str(row.get("video_id") or Path(str(row.get("video_name") or "")).stem).strip()
    if not video_id:
        return None

    image_rel_path = str(row.get("image_rel_path") or "").strip()
    image_gcs_uri = str(row.get("image_gcs_uri") or "").strip()
    image_storage_key = str(row.get("image_storage_key") or "").strip() or image_storage_key_from_uri(image_gcs_uri)
    image_name = PurePosixPath(image_rel_path).name or PurePosixPath(image_storage_key).name
    frame_idx = optional_int(row.get("frame_idx"))
    if frame_idx is None:
        frame_idx = parse_frame_idx(image_name)
    if frame_idx is None:
        return None

    fps = optional_float(row.get("fps")) or DEFAULT_FPS
    frame_seconds = float_value(row.get("frame_sec"), frame_idx / fps if fps else 0.0)
    shot_index = parse_shot_index(row.get("shot_id"), row.get("shot_id_local") or row.get("shot_index"))
    shot_id = normalize_shot_id(video_id, row.get("shot_id"), shot_index)

    if not image_rel_path:
        image_rel_path = f"{video_id}/{image_name}"
    if not image_gcs_uri and image_storage_key:
        image_gcs_uri = f"gs://{bucket_name}/{image_storage_key}"

    return FrameItem(
        dataset_id=str(row.get("dataset_id") or DATASET_ID),
        batch_id=str(row.get("batch_id") or batch_id),
        video_id=video_id,
        video_name=str(row.get("video_name") or f"{video_id}.mp4"),
        video_gcs_uri=str(row.get("video_gcs_uri") or ""),
        shot_id=shot_id,
        shot_index=shot_index,
        shot_start_frame=int_value(row.get("shot_start_frame"), frame_idx),
        shot_end_frame=int_value(row.get("shot_end_frame"), frame_idx),
        shot_start_sec=float_value(row.get("shot_start_sec"), frame_seconds),
        shot_end_sec=float_value(row.get("shot_end_sec"), frame_seconds),
        frame_type=str(row.get("frame_type") or "middle").strip() or "middle",
        frame_idx=frame_idx,
        frame_seconds=frame_seconds,
        keyframe_id=str(row.get("keyframe_id") or f"{video_id}_F{frame_idx:06d}"),
        image_rel_path=image_rel_path,
        image_gcs_uri=image_gcs_uri,
        image_storage_key=image_storage_key,
        fps=fps,
        total_frames=optional_int(row.get("total_frames_opencv")),
        source="shot_segments",
    )


def load_frames_from_shot_segments(client: Any, bucket_name: str, batch_id: str, uri: str) -> list[FrameItem]:
    """Load frame inventory from a shot_segments.csv file."""
    text = read_text(client, uri, bucket_name)
    reader = csv.DictReader(StringIO(text))
    frames = []
    for row in reader:
        item = frame_item_from_shot_row(row, bucket_name, batch_id)
        if item is not None:
            frames.append(item)
    return sorted(frames, key=lambda item: (item.batch_id, item.video_id, item.frame_seconds, item.frame_idx, item.frame_type))


def video_id_from_blob_name(blob_name: str) -> str:
    """Infer video_id from a keyframe object path."""
    parts = PurePosixPath(blob_name).parts
    for part in reversed(parts):
        if part.startswith("video_id="):
            return part.split("=", 1)[1]
    if len(parts) >= 2:
        return parts[-2]
    return ""


def frame_item_from_blob(blob: Any, bucket_name: str, batch_id: str) -> FrameItem | None:
    """Convert one GCS image blob into a FrameItem using filename-derived metadata."""
    path = PurePosixPath(blob.name)
    if path.suffix.lower() not in IMAGE_SUFFIXES:
        return None
    image_name = path.name
    frame_idx = parse_frame_idx(image_name)
    if frame_idx is None:
        return None
    video_id = video_id_from_blob_name(blob.name)
    if not video_id:
        return None
    shot_index = parse_shot_index(image_name, None)
    frame_type = "middle"
    lowered = image_name.lower()
    for candidate in ("first", "middle", "last"):
        if candidate in lowered:
            frame_type = candidate
            break
    frame_seconds = frame_idx / DEFAULT_FPS if DEFAULT_FPS else 0.0
    return FrameItem(
        dataset_id=DATASET_ID,
        batch_id=batch_id,
        video_id=video_id,
        video_name=f"{video_id}.mp4",
        video_gcs_uri="",
        shot_id=f"{video_id}_S{shot_index:04d}",
        shot_index=shot_index,
        shot_start_frame=frame_idx,
        shot_end_frame=frame_idx,
        shot_start_sec=frame_seconds,
        shot_end_sec=frame_seconds,
        frame_type=frame_type,
        frame_idx=frame_idx,
        frame_seconds=frame_seconds,
        keyframe_id=f"{video_id}_F{frame_idx:06d}",
        image_rel_path=f"{video_id}/{image_name}",
        image_gcs_uri=f"gs://{bucket_name}/{blob.name}",
        image_storage_key=blob.name,
        fps=DEFAULT_FPS,
        total_frames=None,
        source="gcs_list",
    )


def list_frames_from_gcs_prefix(client: Any, bucket_name: str, batch_id: str) -> list[FrameItem]:
    """List keyframe images directly from the processed/keyframes GCS prefix."""
    batch_prefix = format_prefix_template(FRAME_PREFIX_TEMPLATE, batch_id).rstrip("/") + "/"
    selected_videos = set(VIDEO_IDS or [])
    prefixes = (
        [f"{batch_prefix}video_id={video_id}/" for video_id in sorted(selected_videos)]
        if selected_videos
        else [batch_prefix]
    )
    frames = []
    for prefix in prefixes:
        for blob in client.list_blobs(bucket_name, prefix=prefix, timeout=GCS_TIMEOUT_SECONDS):
            item = frame_item_from_blob(blob, bucket_name, batch_id)
            if item is not None:
                frames.append(item)
    return sorted(frames, key=lambda item: (item.batch_id, item.video_id, item.frame_seconds, item.frame_idx, item.frame_type))


def filter_inventory(frames: list[FrameItem], max_videos: int | None, max_frames: int | None) -> list[FrameItem]:
    """Apply VIDEO_IDS, MAX_VIDEOS, and MAX_FRAMES limits to an inventory."""
    selected = set(VIDEO_IDS or [])
    if selected:
        frames = [item for item in frames if item.video_id in selected]
    if max_videos:
        keep_videos = set(sorted({item.video_id for item in frames})[: int(max_videos)])
        frames = [item for item in frames if item.video_id in keep_videos]
    frames = sorted(frames, key=lambda item: (item.batch_id, item.video_id, item.frame_seconds, item.frame_idx, item.frame_type))
    if max_frames:
        frames = frames[: int(max_frames)]
    return frames


def load_frame_inventory(
    client: Any,
    bucket_name: str,
    batches: list[str],
    max_videos: int | None = None,
    max_frames: int | None = None,
) -> tuple[list[FrameItem], dict[str, Any]]:
    """Load frame inventory for selected batches and return source diagnostics."""
    all_frames: list[FrameItem] = []
    sources: dict[str, Any] = {}
    for batch_id in batches:
        uri = shot_segments_uri_for_batch(client, bucket_name, batch_id) if USE_SHOT_SEGMENTS_METADATA else ""
        if uri:
            frames = load_frames_from_shot_segments(client, bucket_name, batch_id, uri)
            source = {"kind": "shot_segments", "uri": uri, "frames": len(frames)}
        else:
            frames = list_frames_from_gcs_prefix(client, bucket_name, batch_id)
            source = {
                "kind": "gcs_list",
                "prefix": format_prefix_template(FRAME_PREFIX_TEMPLATE, batch_id),
                "frames": len(frames),
            }
        all_frames.extend(frames)
        sources[batch_id] = source

    # Deduplicate in case a manual URI and prefix overlap.
    deduped: dict[str, FrameItem] = {}
    for item in all_frames:
        deduped[item.keyframe_id] = item
    filtered = filter_inventory(list(deduped.values()), max_videos=max_videos, max_frames=max_frames)
    sources["totals"] = {
        "frames_before_filter": len(deduped),
        "frames_after_filter": len(filtered),
        "videos_after_filter": len({item.video_id for item in filtered}),
    }
    return filtered, sources


# Model

Cell này load OpenCLIP giống backend/query runtime (`ViT-B-32`, `laion2b_s34b_b79k`),
bật các tối ưu GPU của Kaggle, dùng DataLoader CPU workers để decode ảnh và chạy batch inference.

In [ ]:
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

OPENCLIP_MODEL = None
OPENCLIP_PREPROCESS = None
RESOLVED_DEVICE = None


class ImagePathDataset(Dataset):
    """Torch Dataset that opens image paths and applies OpenCLIP preprocessing."""

    def __init__(self, image_paths: list[Path], preprocess: Any) -> None:
        """Store image paths and preprocessing transform."""
        self.image_paths = list(image_paths)
        self.preprocess = preprocess

    def __len__(self) -> int:
        """Return the number of images in the dataset."""
        return len(self.image_paths)

    def __getitem__(self, index: int) -> torch.Tensor:
        """Load one image as RGB and return a preprocessed tensor."""
        with Image.open(self.image_paths[index]) as image:
            return self.preprocess(image.convert("RGB"))


def resolve_device(device: str) -> str:
    """Resolve auto/cuda/cpu device config to a torch device string."""
    if device == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return device


def configure_torch_runtime(device: str) -> None:
    """Enable TensorFloat-32 and cuDNN benchmarking when CUDA is available."""
    if device.startswith("cuda") and torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = bool(ALLOW_TF32)
        torch.backends.cudnn.allow_tf32 = bool(ALLOW_TF32)
        torch.backends.cudnn.benchmark = bool(CUDNN_BENCHMARK)
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass


def autocast_context(device: str):
    """Return the right autocast context for configured precision."""
    if device.startswith("cuda") and EMBED_PRECISION in {"fp16", "bf16"}:
        dtype = torch.float16 if EMBED_PRECISION == "fp16" else torch.bfloat16
        return torch.autocast(device_type="cuda", dtype=dtype)

    class NullContext:
        """No-op context manager for CPU or fp32 inference."""

        def __enter__(self):
            """Enter the no-op context."""
            return None

        def __exit__(self, exc_type, exc, tb):
            """Exit the no-op context without suppressing exceptions."""
            return False

    return NullContext()


def load_openclip_model() -> tuple[Any, Any, str]:
    """Load OpenCLIP image encoder once and return model, preprocess and device."""
    global OPENCLIP_MODEL, OPENCLIP_PREPROCESS, RESOLVED_DEVICE
    if OPENCLIP_MODEL is not None:
        return OPENCLIP_MODEL, OPENCLIP_PREPROCESS, RESOLVED_DEVICE

    import open_clip

    RESOLVED_DEVICE = resolve_device(DEVICE)
    configure_torch_runtime(RESOLVED_DEVICE)
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    model, _, preprocess = open_clip.create_model_and_transforms(
        OPENCLIP_MODEL_NAME,
        pretrained=OPENCLIP_PRETRAINED,
        cache_dir=str(MODEL_CACHE_DIR),
        device=RESOLVED_DEVICE,
    )
    model.eval()
    if TORCH_COMPILE and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
        except Exception as exc:
            print("torch.compile skipped:", exc)
    OPENCLIP_MODEL = model
    OPENCLIP_PREPROCESS = preprocess
    print("Loaded OpenCLIP:", OPENCLIP_MODEL_NAME, OPENCLIP_PRETRAINED, "device:", RESOLVED_DEVICE)
    return OPENCLIP_MODEL, OPENCLIP_PREPROCESS, RESOLVED_DEVICE


def clear_gpu_cache() -> None:
    """Release CUDA cache after a failed batch."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def encode_image_paths(local_paths: list[Path]) -> np.ndarray:
    """Encode local image paths into float32 L2-normalized OpenCLIP vectors."""
    if not local_paths:
        return np.empty((0, 0), dtype=np.float32)
    model, preprocess, device = load_openclip_model()
    worker_count = max(0, int(IMAGE_DECODE_WORKERS))
    loader_kwargs = {
        "batch_size": int(EMBED_BATCH_SIZE),
        "shuffle": False,
        "num_workers": worker_count,
        "pin_memory": bool(PIN_MEMORY and device.startswith("cuda")),
    }
    if worker_count > 0:
        loader_kwargs["prefetch_factor"] = int(DATALOADER_PREFETCH_FACTOR)
        loader_kwargs["persistent_workers"] = bool(PERSISTENT_WORKERS)

    loader = DataLoader(ImagePathDataset(local_paths, preprocess), **loader_kwargs)
    chunks = []
    with torch.inference_mode():
        for images in loader:
            images = images.to(device, non_blocking=True)
            with autocast_context(device):
                vectors = model.encode_image(images, normalize=bool(L2_NORMALIZE))
            chunks.append(vectors.detach().cpu().numpy().astype("float32"))
    return np.concatenate(chunks, axis=0)


def model_metadata() -> dict[str, Any]:
    """Return model metadata written to manifest/report files."""
    return {
        "model_name": OPENCLIP_MODEL_NAME,
        "pretrained": OPENCLIP_PRETRAINED,
        "feature_dir_name": FEATURE_DIR_NAME,
        "model_version": EMBEDDING_MODEL_VERSION,
        "device": resolve_device(DEVICE),
        "precision": EMBED_PRECISION,
        "l2_normalize": bool(L2_NORMALIZE),
    }


def run_parameters() -> dict[str, Any]:
    """Return tunable runtime parameters written to manifest/report files."""
    return {
        "pipeline_batch_size": PIPELINE_BATCH_SIZE,
        "embed_batch_size": EMBED_BATCH_SIZE,
        "download_workers": DOWNLOAD_WORKERS,
        "image_decode_workers": IMAGE_DECODE_WORKERS,
        "upload_workers": UPLOAD_WORKERS,
        "zilliz_shard_size": ZILLIZ_SHARD_SIZE,
        "vector_round_decimals": VECTOR_ROUND_DECIMALS,
        "build_events": BUILD_EVENTS,
        "max_time_gap_sec": MAX_TIME_GAP_SEC,
        "scene_similarity_threshold": SCENE_SIMILARITY_THRESHOLD,
        "max_event_duration_sec": MAX_EVENT_DURATION_SEC,
    }


# Artifacts

Cell này tạo layout output local, ghi JSON/CSV/NPY/JSONL shard, upload cây artifact lên GCS
và sinh các bảng staging cho Supabase/Postgres cùng payload Zilliz.

In [ ]:
@dataclass
class RunLayout:
    """Local folder layout for one feature extraction run."""

    run_id: str
    run_dir: Path
    downloads_dir: Path
    artifacts_dir: Path
    features_dir: Path
    feature_dir: Path
    map_keyframes_dir: Path
    events_dir: Path
    map_event_dir: Path
    postgres_dir: Path
    zilliz_keyframes_dir: Path
    zilliz_events_dir: Path
    log_path: Path
    gcs_prefix: str


class JsonlShardWriter:
    """Streaming JSONL writer that rotates files after a fixed number of records."""

    def __init__(self, output_dir: Path, basename: str, shard_size: int) -> None:
        """Create a writer for sharded JSONL files."""
        self.output_dir = output_dir
        self.basename = basename
        self.shard_size = int(shard_size)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.shard_index = 0
        self.records_in_current = 0
        self.total_records = 0
        self.current_handle = None
        self.parts: list[dict[str, Any]] = []

    def _open_next(self) -> None:
        """Open the next shard file for appending JSONL records."""
        if self.current_handle is not None:
            self.current_handle.close()
        filename = f"{self.basename}-part-{self.shard_index:06d}.jsonl"
        path = self.output_dir / filename
        self.current_handle = path.open("w", encoding="utf-8")
        self.parts.append({"part_index": self.shard_index, "path": str(path), "records": 0})
        self.shard_index += 1
        self.records_in_current = 0

    def write(self, row: dict[str, Any]) -> None:
        """Write one JSONL row and rotate the shard if needed."""
        if self.current_handle is None or self.records_in_current >= self.shard_size:
            self._open_next()
        assert self.current_handle is not None
        self.current_handle.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")
        self.records_in_current += 1
        self.total_records += 1
        self.parts[-1]["records"] += 1

    def close(self) -> list[dict[str, Any]]:
        """Close the current shard and return part metadata."""
        if self.current_handle is not None:
            self.current_handle.close()
            self.current_handle = None
        return list(self.parts)


def make_run_layout(run_id: str, bucket_name: str) -> RunLayout:
    """Create all local output directories for a run."""
    run_dir = LOCAL_RUN_ROOT / run_id
    artifacts_dir = run_dir / "artifacts"
    features_dir = run_dir / "features"
    layout = RunLayout(
        run_id=run_id,
        run_dir=run_dir,
        downloads_dir=run_dir / "downloads",
        artifacts_dir=artifacts_dir,
        features_dir=features_dir,
        feature_dir=features_dir / FEATURE_DIR_NAME,
        map_keyframes_dir=features_dir / "map-keyframes",
        events_dir=features_dir / "events",
        map_event_dir=features_dir / "map-event",
        postgres_dir=run_dir / "postgres",
        zilliz_keyframes_dir=run_dir / "zilliz" / ZILLIZ_KEYFRAME_COLLECTION,
        zilliz_events_dir=run_dir / "zilliz" / ZILLIZ_EVENT_COLLECTION,
        log_path=run_dir / "run.log",
        gcs_prefix=(
            f"{normalize_prefix(OUTPUT_PREFIX)}/dataset={DATASET_ID}/"
            f"profile={PROFILE_VERSION}/run_id={run_id}/"
        ),
    )
    for path in [
        layout.downloads_dir,
        layout.artifacts_dir,
        layout.feature_dir,
        layout.map_keyframes_dir,
        layout.events_dir,
        layout.map_event_dir,
        layout.postgres_dir,
        layout.zilliz_keyframes_dir,
        layout.zilliz_events_dir,
    ]:
        path.mkdir(parents=True, exist_ok=True)
    return layout


def write_json(path: Path, payload: dict[str, Any]) -> None:
    """Write a dictionary as pretty UTF-8 JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")


def write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    """Write dictionaries as UTF-8 JSONL."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")


def write_csv(path: Path, rows: list[dict[str, Any]], columns: list[str]) -> None:
    """Write rows to CSV with stable column order, even when rows are empty."""
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=columns).to_csv(path, index=False)


def vector_to_json(vector: np.ndarray) -> list[float]:
    """Convert a numpy vector to a compact JSON-safe list."""
    if VECTOR_ROUND_DECIMALS is not None:
        vector = np.round(vector.astype(np.float32), int(VECTOR_ROUND_DECIMALS))
    return vector.astype(float).tolist()


def local_download_path(layout: RunLayout, item: FrameItem) -> Path:
    """Return the local path used to cache/download one frame."""
    return layout.downloads_dir / item.batch_id / item.video_id / item.image_name


def download_frame_batch(client: Any, bucket_name: str, layout: RunLayout, frames: list[FrameItem]) -> tuple[list[Path], int]:
    """Download a batch of frame images from GCS using thread workers."""
    bucket = client.bucket(bucket_name)
    local_paths = [local_download_path(layout, item) for item in frames]

    def download_one(item: FrameItem, destination: Path) -> Path:
        """Download one frame unless a reusable local copy already exists."""
        destination.parent.mkdir(parents=True, exist_ok=True)
        if REUSE_LOCAL_DOWNLOADS and destination.exists() and destination.stat().st_size > 0:
            return destination
        bucket.blob(item.image_storage_key).download_to_filename(str(destination), timeout=GCS_TIMEOUT_SECONDS)
        return destination

    with ThreadPoolExecutor(max_workers=max(1, int(DOWNLOAD_WORKERS))) as pool:
        futures = [pool.submit(download_one, item, path) for item, path in zip(frames, local_paths)]
        for future in as_completed(futures):
            future.result()
    return local_paths, sum(path.stat().st_size for path in local_paths if path.exists())


def remove_local_downloads(paths: list[Path]) -> None:
    """Remove downloaded frame files after each batch when configured."""
    if not DELETE_LOCAL_AFTER_BATCH:
        return
    for path in paths:
        path.unlink(missing_ok=True)


def build_keyframe_zilliz_record(
    item: FrameItem,
    vector: np.ndarray,
    embedding_index_0: int,
    map_n: int,
    dataset_db_id: str,
) -> dict[str, Any]:
    """Build one keyframe vector payload compatible with Zilliz/Milvus dynamic fields."""
    return {
        "id": item.keyframe_id,
        "vector": vector_to_json(vector),
        "keyframe_id": item.keyframe_id,
        "video_id": item.video_id,
        "dataset_id": dataset_db_id,
        "dataset_code": DATASET_CODE,
        "batch_id": item.batch_id,
        "shot_id": item.shot_id,
        "frame_idx": int(item.frame_idx),
        "frame_seconds": float(item.frame_seconds),
        "timestamp_ms": int(item.timestamp_ms),
        "frame_type": item.frame_type,
        "embedding_index_0": int(embedding_index_0),
        "map_n": int(map_n),
        "image_uri": item.image_gcs_uri,
        "image_storage_key": item.image_storage_key,
        "model_version": EMBEDDING_MODEL_VERSION,
        "profile_version": PROFILE_VERSION,
    }


def upload_file(client: Any, bucket_name: str, local_path: Path, blob_name: str) -> dict[str, Any]:
    """Upload one local file to GCS and return timing metadata."""
    content_type = "application/octet-stream"
    suffix = local_path.suffix.lower()
    if suffix == ".json":
        content_type = "application/json"
    elif suffix == ".jsonl":
        content_type = "application/jsonl"
    elif suffix == ".csv":
        content_type = "text/csv"
    elif suffix == ".npy":
        content_type = "application/octet-stream"
    elif suffix == ".md":
        content_type = "text/markdown"
    started = time.perf_counter()
    client.bucket(bucket_name).blob(normalize_prefix(blob_name)).upload_from_filename(
        str(local_path),
        content_type=content_type,
        timeout=GCS_TIMEOUT_SECONDS,
    )
    return {
        "local_path": str(local_path),
        "gcs_uri": f"gs://{bucket_name}/{normalize_prefix(blob_name)}",
        "bytes": local_path.stat().st_size,
        "duration_ms": millis_since(started),
    }


def upload_tree(client: Any, bucket_name: str, layout: RunLayout, logger: logging.Logger) -> list[dict[str, Any]]:
    """Upload the run artifact tree to GCS, skipping temporary downloads."""
    files = [
        path
        for path in sorted(layout.run_dir.rglob("*"))
        if path.is_file() and layout.downloads_dir not in path.parents
    ]
    uploaded: list[dict[str, Any]] = []
    total = len(files)
    if total == 0:
        return uploaded

    def upload_one(path: Path) -> dict[str, Any]:
        """Upload one artifact preserving its relative path under the run prefix."""
        rel = path.relative_to(layout.run_dir).as_posix()
        return upload_file(client, bucket_name, path, layout.gcs_prefix + rel)

    with ThreadPoolExecutor(max_workers=max(1, int(UPLOAD_WORKERS))) as pool:
        futures = {pool.submit(upload_one, path): path for path in files}
        for index, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            uploaded.append(result)
            percent = 100.0 * index / max(total, 1)
            logger.info(
                "[upload %d/%d %.2f%%] %s bytes=%d duration_ms=%d",
                index,
                total,
                percent,
                result["gcs_uri"],
                result["bytes"],
                result["duration_ms"],
            )
    return uploaded


def write_inventory_artifacts(layout: RunLayout, frames: list[FrameItem], sources: dict[str, Any]) -> None:
    """Write processing manifest and reconstructed shot_segments.csv for audit/import."""
    write_jsonl(layout.artifacts_dir / "processing_manifest.jsonl", [asdict(item) for item in frames])
    write_json(layout.artifacts_dir / "input_sources.json", sources)
    shot_columns = [
        "dataset_id",
        "batch_id",
        "video_id",
        "video_name",
        "video_gcs_uri",
        "shot_id",
        "shot_id_local",
        "shot_start_frame",
        "shot_end_frame",
        "shot_start_sec",
        "shot_end_sec",
        "frame_type",
        "frame_idx",
        "frame_sec",
        "keyframe_id",
        "image_rel_path",
        "image_gcs_uri",
        "image_storage_key",
        "boundary_threshold",
        "min_shot_len",
        "saved",
        "fps",
        "total_frames_opencv",
        "profile_version",
        "run_id",
    ]
    rows = [
        {
            "dataset_id": item.dataset_id,
            "batch_id": item.batch_id,
            "video_id": item.video_id,
            "video_name": item.video_name,
            "video_gcs_uri": item.video_gcs_uri,
            "shot_id": item.shot_id,
            "shot_id_local": item.shot_index,
            "shot_start_frame": item.shot_start_frame,
            "shot_end_frame": item.shot_end_frame,
            "shot_start_sec": item.shot_start_sec,
            "shot_end_sec": item.shot_end_sec,
            "frame_type": item.frame_type,
            "frame_idx": item.frame_idx,
            "frame_sec": item.frame_seconds,
            "keyframe_id": item.keyframe_id,
            "image_rel_path": item.image_rel_path,
            "image_gcs_uri": item.image_gcs_uri,
            "image_storage_key": item.image_storage_key,
            "boundary_threshold": "",
            "min_shot_len": "",
            "saved": True,
            "fps": item.fps if item.fps is not None else "",
            "total_frames_opencv": item.total_frames if item.total_frames is not None else "",
            "profile_version": PROFILE_VERSION,
            "run_id": layout.run_id,
        }
        for item in frames
    ]
    write_csv(layout.artifacts_dir / "shot_segments.csv", rows, shot_columns)


# Supabase/Postgres Export

Cell này sinh các CSV staging bám theo schema SQLAlchemy trong backend:
`datasets`, `videos`, `shots`, `keyframes`, `model_registry`, `index_builds`.
Vector không được nhét vào Postgres mặc định; vector đi sang Zilliz/Milvus.

In [ ]:
def build_video_summary_rows(
    frames_by_video: dict[str, list[FrameItem]],
    embedding_shapes: dict[str, tuple[int, int]],
    layout: RunLayout,
) -> list[dict[str, Any]]:
    """Build per-video summary rows compatible with legacy import scripts."""
    rows = []
    for video_id, items in sorted(frames_by_video.items()):
        seconds = max([item.shot_end_sec for item in items] + [item.frame_seconds])
        shape = embedding_shapes.get(video_id, (len(items), 0))
        rows.append(
            {
                "video_id": video_id,
                "num_keyframes": len(items),
                "seconds": round(float(seconds), 6),
                "embedding_shape": f"{shape[0]}x{shape[1]}",
                "feature_path": str(layout.feature_dir / f"{video_id}.npy"),
                "map_path": str(layout.map_keyframes_dir / f"{video_id}.csv"),
            }
        )
    return rows


def write_per_video_summary(layout: RunLayout, rows: list[dict[str, Any]]) -> None:
    """Write per_video_summary.csv in the demo-compatible root format."""
    write_csv(
        layout.run_dir / "per_video_summary.csv",
        rows,
        ["video_id", "num_keyframes", "seconds", "embedding_shape", "feature_path", "map_path"],
    )


def build_postgres_base_exports(
    layout: RunLayout,
    frames_by_video: dict[str, list[FrameItem]],
    map_n_by_keyframe: dict[str, int],
    embedding_shapes: dict[str, tuple[int, int]],
    bucket_name: str,
    run_id: str,
) -> dict[str, Path]:
    """Write base PostgreSQL/Supabase CSV files for dataset, video, shot and keyframe rows."""
    dataset_db_id = resolve_dataset_db_id()
    root_uri = f"gs://{bucket_name}/{normalize_prefix(KEYFRAMES_PREFIX)}/dataset={DATASET_ID}"
    datasets_rows = [
        {
            "dataset_id": dataset_db_id,
            "dataset_code": DATASET_CODE,
            "name": DATASET_NAME,
            "version": DATASET_VERSION,
            "root_uri": root_uri,
            "status": "READY",
        }
    ]
    write_csv(
        layout.postgres_dir / "datasets.csv",
        datasets_rows,
        ["dataset_id", "dataset_code", "name", "version", "root_uri", "status"],
    )

    video_rows = []
    shot_rows_by_id: dict[str, dict[str, Any]] = {}
    keyframe_rows = []
    for video_id, items in sorted(frames_by_video.items()):
        first = items[0]
        seconds = max([item.shot_end_sec for item in items] + [item.frame_seconds])
        shape = embedding_shapes.get(video_id, (len(items), 0))
        video_rows.append(
            {
                "video_id": video_id,
                "dataset_id": dataset_db_id,
                "video_code": video_id,
                "video_name": first.video_name or f"{video_id}.mp4",
                "uri": first.video_gcs_uri or f"{root_uri}/batch={first.batch_id}/video_id={video_id}",
                "source_video_path": first.video_gcs_uri,
                "fps": first.fps if first.fps is not None else "",
                "duration_seconds": round(float(seconds), 6),
                "duration_ms": int(float(seconds) * 1000),
                "width": "",
                "height": "",
                "num_keyframes": len(items),
                "embedding_shape": f"{shape[0]}x{shape[1]}",
                "source_feature_path": str(layout.feature_dir / f"{video_id}.npy"),
                "source_map_path": str(layout.map_keyframes_dir / f"{video_id}.csv"),
                "extra_metadata": json.dumps(
                    {
                        "feature_model_version": EMBEDDING_MODEL_VERSION,
                        "run_id": run_id,
                        "profile_version": PROFILE_VERSION,
                    },
                    ensure_ascii=False,
                ),
            }
        )
        for item in items:
            if item.shot_id not in shot_rows_by_id:
                shot_rows_by_id[item.shot_id] = {
                    "shot_id": item.shot_id,
                    "video_id": item.video_id,
                    "shot_index": item.shot_index,
                    "start_frame": item.shot_start_frame,
                    "end_frame": item.shot_end_frame,
                    "start_seconds": item.shot_start_sec,
                    "end_seconds": item.shot_end_sec,
                    "boundary_threshold": "",
                }
            map_n = map_n_by_keyframe.get(item.keyframe_id)
            keyframe_rows.append(
                {
                    "keyframe_id": item.keyframe_id,
                    "video_id": item.video_id,
                    "shot_id": item.shot_id,
                    "frame_idx": item.frame_idx,
                    "frame_seconds": item.frame_seconds,
                    "timestamp_ms": item.timestamp_ms,
                    "frame_type": item.frame_type,
                    "map_n": map_n if map_n is not None else "",
                    "embedding_index_0": (map_n - 1) if map_n else "",
                    "image_rel_path": item.image_rel_path,
                    "image_storage_key": item.image_storage_key,
                    "image_url": public_url(bucket_name, item.image_storage_key),
                    "image_uri": item.image_gcs_uri,
                    "thumbnail_uri": public_url(bucket_name, item.image_storage_key),
                    "dedup_group_id": "",
                    "quality_score": 1.0,
                    "is_media_present": True,
                }
            )

    write_csv(
        layout.postgres_dir / "videos.csv",
        video_rows,
        [
            "video_id",
            "dataset_id",
            "video_code",
            "video_name",
            "uri",
            "source_video_path",
            "fps",
            "duration_seconds",
            "duration_ms",
            "width",
            "height",
            "num_keyframes",
            "embedding_shape",
            "source_feature_path",
            "source_map_path",
            "extra_metadata",
        ],
    )
    write_csv(
        layout.postgres_dir / "shots.csv",
        list(shot_rows_by_id.values()),
        [
            "shot_id",
            "video_id",
            "shot_index",
            "start_frame",
            "end_frame",
            "start_seconds",
            "end_seconds",
            "boundary_threshold",
        ],
    )
    write_csv(
        layout.postgres_dir / "keyframes.csv",
        keyframe_rows,
        [
            "keyframe_id",
            "video_id",
            "shot_id",
            "frame_idx",
            "frame_seconds",
            "timestamp_ms",
            "frame_type",
            "map_n",
            "embedding_index_0",
            "image_rel_path",
            "image_storage_key",
            "image_url",
            "image_uri",
            "thumbnail_uri",
            "dedup_group_id",
            "quality_score",
            "is_media_present",
        ],
    )

    model_registry_rows = [
        {
            "id": str(uuid.uuid5(uuid.NAMESPACE_URL, f"model-registry:{EMBEDDING_MODEL_VERSION}:image_embedding")),
            "name": EMBEDDING_MODEL_VERSION,
            "task": "multimodal_embedding",
            "provider": "open_clip",
            "checkpoint_uri": f"{OPENCLIP_MODEL_NAME}:{OPENCLIP_PRETRAINED}",
            "config": json.dumps(model_metadata(), ensure_ascii=False),
            "status": "READY",
        }
    ]
    write_csv(
        layout.postgres_dir / "model_registry.csv",
        model_registry_rows,
        ["id", "name", "task", "provider", "checkpoint_uri", "config", "status"],
    )

    index_build_rows = [
        {
            "id": str(uuid.uuid5(uuid.NAMESPACE_URL, f"index-build:{dataset_db_id}:{ZILLIZ_KEYFRAME_COLLECTION}:{EMBEDDING_MODEL_VERSION}:{run_id}")),
            "dataset_id": dataset_db_id,
            "index_type": "milvus",
            "collection_name": ZILLIZ_KEYFRAME_COLLECTION,
            "model_name": OPENCLIP_MODEL_NAME,
            "model_version": EMBEDDING_MODEL_VERSION,
            "status": "READY",
            "stats": json.dumps(
                {
                    "vectors": sum(len(items) for items in frames_by_video.values()),
                    "metric_type": "COSINE",
                    "l2_normalize": bool(L2_NORMALIZE),
                    "run_id": run_id,
                },
                ensure_ascii=False,
            ),
            "completed_at": utc_now(),
        }
    ]
    write_csv(
        layout.postgres_dir / "index_builds.csv",
        index_build_rows,
        ["id", "dataset_id", "index_type", "collection_name", "model_name", "model_version", "status", "stats", "completed_at"],
    )

    return {
        "datasets": layout.postgres_dir / "datasets.csv",
        "videos": layout.postgres_dir / "videos.csv",
        "shots": layout.postgres_dir / "shots.csv",
        "keyframes": layout.postgres_dir / "keyframes.csv",
        "model_registry": layout.postgres_dir / "model_registry.csv",
        "index_builds": layout.postgres_dir / "index_builds.csv",
    }


# Event Embeddings

Cell này port logic từ `event-embeddings.ipynb`: gom keyframe liên tiếp theo khoảng
cách thời gian và cosine similarity, sau đó ghi `features/events`, `features/map-event`,
CSV Supabase cho `events/event_keyframes`, và JSONL Zilliz cho event vectors.

In [ ]:
def normalize_vectors(vectors: np.ndarray) -> np.ndarray:
    """L2-normalize rows of a float32 matrix."""
    vectors = vectors.astype(np.float32, copy=False)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(norms, 1e-12)


def flush_event(
    video_id: str,
    key_map: pd.DataFrame,
    embeddings: np.ndarray,
    indices: list[int],
    event_index: int,
) -> tuple[dict[str, Any], np.ndarray]:
    """Create one event row and normalized event vector from buffered keyframe indices."""
    idx = np.asarray(indices, dtype=int)
    event_vec = embeddings[idx].mean(axis=0)
    event_vec = event_vec / max(float(np.linalg.norm(event_vec)), 1e-12)
    start_row = key_map.iloc[indices[0]]
    end_row = key_map.iloc[indices[-1]]
    keyframe_ns = key_map.iloc[idx]["n"].astype(int).tolist()
    keyframe_ids = key_map.iloc[idx]["keyframe_id"].astype(str).tolist()
    shot_ids = sorted(set(key_map.iloc[idx]["shot_id"].astype(str).tolist()))
    event_id = f"{video_id}_E{event_index:06d}"
    row = {
        "event_id": event_id,
        "event_embedding_index": event_index,
        "video_id": video_id,
        "start_n": int(start_row["n"]),
        "end_n": int(end_row["n"]),
        "start_sec": float(start_row["pts_time"]),
        "end_sec": float(end_row["pts_time"]),
        "start_frame": int(start_row["frame_idx"]),
        "end_frame": int(end_row["frame_idx"]),
        "representative_keyframe_id": str(start_row["keyframe_id"]),
        "keyframe_ns": " ".join(map(str, keyframe_ns)),
        "keyframe_ids": " ".join(keyframe_ids),
        "shot_ids": " ".join(shot_ids),
        "n_keyframes": int(len(idx)),
    }
    return row, event_vec.astype(np.float32)


def segment_events_for_video(video_id: str, embeddings: np.ndarray, key_map: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Segment one video's keyframes into temporal/semantic events."""
    if embeddings.size == 0 or key_map.empty:
        return pd.DataFrame(), np.empty((0, 0), dtype=np.float32)

    key_map = key_map.copy()
    key_map["n"] = key_map["n"].astype(int)
    key_map["pts_time"] = key_map["pts_time"].astype(float)
    key_map["frame_idx"] = key_map["frame_idx"].astype(int)
    key_map = key_map.sort_values(["pts_time", "frame_idx", "n"]).reset_index(drop=True)
    embeddings = embeddings[key_map["n"].to_numpy(dtype=int) - 1]
    embeddings = normalize_vectors(embeddings)

    events: list[dict[str, Any]] = []
    event_vectors: list[np.ndarray] = []
    current_indices = [0]
    current_vector = embeddings[0].copy()

    for idx in range(1, len(key_map)):
        prev_time = float(key_map.loc[idx - 1, "pts_time"])
        cur_time = float(key_map.loc[idx, "pts_time"])
        event_start_time = float(key_map.loc[current_indices[0], "pts_time"])
        time_gap = cur_time - prev_time
        event_duration_if_added = cur_time - event_start_time
        similarity = float(np.dot(current_vector, embeddings[idx]))

        should_split = False
        if time_gap > float(MAX_TIME_GAP_SEC):
            should_split = True
        if similarity < float(SCENE_SIMILARITY_THRESHOLD):
            should_split = True
        if MAX_EVENT_DURATION_SEC is not None and event_duration_if_added > float(MAX_EVENT_DURATION_SEC):
            should_split = True

        if should_split:
            row, vector = flush_event(video_id, key_map, embeddings, current_indices, len(events))
            events.append(row)
            event_vectors.append(vector)
            current_indices = [idx]
            current_vector = embeddings[idx].copy()
        else:
            current_indices.append(idx)
            current_vector = embeddings[current_indices].mean(axis=0)
            current_vector = current_vector / max(float(np.linalg.norm(current_vector)), 1e-12)

    row, vector = flush_event(video_id, key_map, embeddings, current_indices, len(events))
    events.append(row)
    event_vectors.append(vector)
    return pd.DataFrame(events), np.stack(event_vectors).astype(np.float32)


def build_event_zilliz_record(
    event_row: dict[str, Any],
    vector: np.ndarray,
    dataset_db_id: str,
) -> dict[str, Any]:
    """Build one event vector payload compatible with Zilliz/Milvus dynamic fields."""
    return {
        "id": event_row["event_id"],
        "vector": vector_to_json(vector),
        "event_id": event_row["event_id"],
        "video_id": event_row["video_id"],
        "dataset_id": dataset_db_id,
        "dataset_code": DATASET_CODE,
        "event_order": int(event_row["event_embedding_index"]),
        "embedding_index_0": int(event_row["event_embedding_index"]),
        "start_seconds": float(event_row["start_sec"]),
        "end_seconds": float(event_row["end_sec"]),
        "start_frame": int(event_row["start_frame"]),
        "end_frame": int(event_row["end_frame"]),
        "representative_keyframe_id": event_row["representative_keyframe_id"],
        "n_keyframes": int(event_row["n_keyframes"]),
        "model_version": EMBEDDING_MODEL_VERSION,
        "segmentation_version": SEGMENTATION_VERSION,
    }


def build_event_outputs(
    layout: RunLayout,
    frames_by_video: dict[str, list[FrameItem]],
    logger: logging.Logger,
) -> dict[str, Any]:
    """Build event embeddings, map-event CSV files, Postgres CSVs, and Zilliz event shards."""
    dataset_db_id = resolve_dataset_db_id()
    event_writer = JsonlShardWriter(layout.zilliz_events_dir, ZILLIZ_EVENT_COLLECTION, ZILLIZ_SHARD_SIZE)
    pg_event_rows: list[dict[str, Any]] = []
    pg_event_keyframe_rows: list[dict[str, Any]] = []
    event_summary_rows: list[dict[str, Any]] = []
    total_events = 0

    for video_id in sorted(frames_by_video):
        feature_path = layout.feature_dir / f"{video_id}.npy"
        map_path = layout.map_keyframes_dir / f"{video_id}.csv"
        if not feature_path.exists() or not map_path.exists():
            logger.warning("skip events for video_id=%s because feature/map file is missing", video_id)
            continue
        embeddings = np.load(feature_path).astype(np.float32)
        key_map = pd.read_csv(map_path)
        event_map, event_embeddings = segment_events_for_video(video_id, embeddings, key_map)
        if event_map.empty:
            continue
        np.save(layout.events_dir / f"{video_id}.npy", event_embeddings)
        event_map.to_csv(layout.map_event_dir / f"{video_id}.csv", index=False)
        total_events += len(event_map)

        for row_dict, vector in zip(event_map.to_dict("records"), event_embeddings):
            event_writer.write(build_event_zilliz_record(row_dict, vector, dataset_db_id))
            pg_event_rows.append(
                {
                    "event_id": row_dict["event_id"],
                    "video_id": row_dict["video_id"],
                    "embedding_index_0": int(row_dict["event_embedding_index"]),
                    "start_seconds": float(row_dict["start_sec"]),
                    "end_seconds": float(row_dict["end_sec"]),
                    "start_frame": int(row_dict["start_frame"]),
                    "end_frame": int(row_dict["end_frame"]),
                    "representative_keyframe_id": row_dict["representative_keyframe_id"],
                    "n_shots": len(row_dict["shot_ids"].split()) if row_dict["shot_ids"] else "",
                    "n_keyframes": int(row_dict["n_keyframes"]),
                    "shot_ids_raw": row_dict["shot_ids"],
                    "keyframe_embedding_indices_raw": row_dict["keyframe_ns"],
                    "start_frame_idx": int(row_dict["start_frame"]),
                    "end_frame_idx": int(row_dict["end_frame"]),
                    "representative_frame_id": row_dict["representative_keyframe_id"],
                    "title": row_dict["event_id"],
                    "description": "",
                    "event_order": int(row_dict["event_embedding_index"]),
                    "segmentation_version": SEGMENTATION_VERSION,
                }
            )
            for seq_no, keyframe_id in enumerate(row_dict["keyframe_ids"].split()):
                pg_event_keyframe_rows.append(
                    {
                        "event_id": row_dict["event_id"],
                        "seq_no": seq_no,
                        "keyframe_id": keyframe_id,
                        "keyframe_embedding_index_0": int(row_dict["keyframe_ns"].split()[seq_no]) - 1,
                    }
                )
        event_summary_rows.append(
            {
                "video_id": video_id,
                "num_keyframes": int(len(key_map)),
                "num_events": int(len(event_map)),
                "embedding_dim": int(event_embeddings.shape[1]),
                "event_feature_path": str(layout.events_dir / f"{video_id}.npy"),
                "event_map_path": str(layout.map_event_dir / f"{video_id}.csv"),
            }
        )
        logger.info("events video_id=%s keyframes=%d events=%d", video_id, len(key_map), len(event_map))

    event_parts = event_writer.close()
    write_csv(
        layout.postgres_dir / "events.csv",
        pg_event_rows,
        [
            "event_id",
            "video_id",
            "embedding_index_0",
            "start_seconds",
            "end_seconds",
            "start_frame",
            "end_frame",
            "representative_keyframe_id",
            "n_shots",
            "n_keyframes",
            "shot_ids_raw",
            "keyframe_embedding_indices_raw",
            "start_frame_idx",
            "end_frame_idx",
            "representative_frame_id",
            "title",
            "description",
            "event_order",
            "segmentation_version",
        ],
    )
    write_csv(
        layout.postgres_dir / "event_keyframes.csv",
        pg_event_keyframe_rows,
        ["event_id", "seq_no", "keyframe_id", "keyframe_embedding_index_0"],
    )
    write_csv(
        layout.features_dir / "event_summary.csv",
        event_summary_rows,
        ["video_id", "num_keyframes", "num_events", "embedding_dim", "event_feature_path", "event_map_path"],
    )
    return {
        "events": total_events,
        "event_keyframes": len(pg_event_keyframe_rows),
        "event_zilliz_parts": event_parts,
        "event_summary_rows": event_summary_rows,
    }


# Run Helpers

Cell này điều phối toàn bộ quá trình: dry run, smoke test, demo 1 batch và full run.
Log sẽ hiển thị số frame trong batch, phần trăm hoàn thành, thời gian download/inference/write,
tốc độ frame/giây, và tiến độ upload artifact lên GCS.

In [ ]:
def group_frames_by_video(frames: list[FrameItem]) -> dict[str, list[FrameItem]]:
    """Group frame items by video id while preserving sorted order within each video."""
    grouped: dict[str, list[FrameItem]] = defaultdict(list)
    for item in sorted(frames, key=lambda x: (x.video_id, x.frame_seconds, x.frame_idx, x.frame_type)):
        grouped[item.video_id].append(item)
    return dict(grouped)


def chunked(items: list[Any], size: int) -> Iterable[list[Any]]:
    """Yield fixed-size chunks from a list."""
    if size <= 0:
        raise ValueError("Chunk size must be positive.")
    for start in range(0, len(items), size):
        yield items[start : start + size]


def preview_plan(
    batches: Any = None,
    max_videos: int | None = None,
    max_frames: int | None = DRY_RUN_MAX_FRAMES,
) -> dict[str, Any]:
    """List planned frames without downloading images or loading the model."""
    client = make_storage_client()
    bucket_name = resolve_gcs_bucket()
    batch_ids = selected_batches(BATCHES if batches is None else batches)
    frames, sources = load_frame_inventory(client, bucket_name, batch_ids, max_videos=max_videos, max_frames=max_frames)
    report = {
        "mode": "dry_run",
        "bucket": bucket_name,
        "batches": batch_ids,
        "frames": len(frames),
        "videos": len({item.video_id for item in frames}),
        "sources": sources,
        "sample": [asdict(item) for item in frames[: min(len(frames), 10)]],
        "created_at": utc_now(),
    }
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report


def save_video_embeddings(
    layout: RunLayout,
    video_id: str,
    video_items: list[FrameItem],
    video_embeddings: list[np.ndarray],
) -> tuple[tuple[int, int], list[dict[str, Any]]]:
    """Save per-video NPY embeddings and map-keyframes CSV."""
    if not video_embeddings:
        return (0, 0), []
    matrix = np.concatenate(video_embeddings, axis=0).astype(np.float32)
    np.save(layout.feature_dir / f"{video_id}.npy", matrix)
    rows = []
    for n, item in enumerate(video_items, start=1):
        rows.append(
            {
                "n": n,
                "keyframe_id": item.keyframe_id,
                "video_id": item.video_id,
                "shot_id": item.shot_id,
                "shot_index": item.shot_index,
                "frame_idx": item.frame_idx,
                "pts_time": item.frame_seconds,
                "frame_seconds": item.frame_seconds,
                "timestamp_ms": item.timestamp_ms,
                "frame_type": item.frame_type,
                "fps": item.fps if item.fps is not None else "",
                "image_name": item.image_name,
                "image_rel_path": item.image_rel_path,
                "image_gcs_uri": item.image_gcs_uri,
                "image_storage_key": item.image_storage_key,
                "embedding_index_0": n - 1,
                "model_version": EMBEDDING_MODEL_VERSION,
            }
        )
    pd.DataFrame(rows).to_csv(layout.map_keyframes_dir / f"{video_id}.csv", index=False)
    return (int(matrix.shape[0]), int(matrix.shape[1])), rows


def write_run_report(layout: RunLayout, summary: dict[str, Any]) -> None:
    """Write a human-readable markdown report for the current run."""
    lines = [
        f"# {TASK_NAME} Run Report",
        "",
        f"- Run id: `{summary['run_id']}`",
        f"- Mode: `{summary['mode']}`",
        f"- Status: `{summary['status']}`",
        f"- Frames planned: `{summary['totals']['planned_frames']}`",
        f"- Frames processed: `{summary['totals']['processed_frames']}`",
        f"- Failed frames: `{summary['totals']['failed_frames']}`",
        f"- Videos: `{summary['totals']['videos']}`",
        f"- Keyframe vectors: `{summary['totals']['keyframe_vectors']}`",
        f"- Events: `{summary['totals'].get('events', 0)}`",
        f"- Elapsed seconds: `{summary['timing']['elapsed_seconds']}`",
        f"- Frames/sec: `{summary['timing']['frames_per_second']}`",
        "",
        "## Key Artifacts",
        "",
        "- `features/<model>/<video_id>.npy`",
        "- `features/map-keyframes/<video_id>.csv`",
        "- `postgres/*.csv`",
        "- `zilliz/keyframe_embeddings/*.jsonl`",
        "- `artifacts/summary.json`",
        "- `run.log`",
    ]
    if BUILD_EVENTS:
        lines.extend(
            [
                "- `features/events/<video_id>.npy`",
                "- `features/map-event/<video_id>.csv`",
                "- `zilliz/event_embeddings/*.jsonl`",
            ]
        )
    (layout.artifacts_dir / "report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def run_extraction(
    mode: str,
    batches: Any,
    max_videos: int | None = None,
    max_frames: int | None = None,
    max_batches: int | None = None,
    skip_frames: int = 0,
    upload_artifacts: bool = False,
) -> dict[str, Any]:
    """Run feature extraction for smoke, demo, or full mode."""
    client = make_storage_client()
    bucket_name = resolve_gcs_bucket()
    run_id = make_run_id(mode)
    layout = make_run_layout(run_id, bucket_name)
    logger = setup_console_logger(f"{TASK_NAME}.{run_id}", layout.log_path)
    dataset_db_id = resolve_dataset_db_id()

    selected = selected_batches(batches)
    load_max_frames = (int(skip_frames) + int(max_frames)) if max_frames is not None else None
    frames, sources = load_frame_inventory(client, bucket_name, selected, max_videos=max_videos, max_frames=load_max_frames)
    loaded_frames_before_skip = len(frames)
    if skip_frames:
        frames = frames[int(skip_frames) :]
    if max_frames is not None:
        frames = frames[: int(max_frames)]
    if not frames:
        raise RuntimeError("No frames found. Check batch, frame prefix, shot_segments URI, and GCS credentials.")
    grouped = group_frames_by_video(frames)
    total_batches = sum(math.ceil(len(items) / PIPELINE_BATCH_SIZE) for items in grouped.values())
    if max_batches:
        total_batches = min(total_batches, int(max_batches))

    write_inventory_artifacts(layout, frames, sources)
    logger.info(
        "run start mode=%s run_id=%s bucket=%s batches=%s videos=%d frames=%d total_batches=%d",
        mode,
        run_id,
        bucket_name,
        selected,
        len(grouped),
        len(frames),
        total_batches,
    )
    load_openclip_model()

    zilliz_writer = JsonlShardWriter(layout.zilliz_keyframes_dir, ZILLIZ_KEYFRAME_COLLECTION, ZILLIZ_SHARD_SIZE)
    errors: list[dict[str, Any]] = []
    batch_metrics: list[dict[str, Any]] = []
    embedding_shapes: dict[str, tuple[int, int]] = {}
    map_n_by_keyframe: dict[str, int] = {}
    processed_items_by_video: dict[str, list[FrameItem]] = {}
    processed_frames = 0
    failed_frames = 0
    global_batch_index = 0
    started = time.perf_counter()

    try:
        for video_index, (video_id, video_items) in enumerate(grouped.items(), start=1):
            logger.info(
                "[video %d/%d] start video_id=%s frames=%d",
                video_index,
                len(grouped),
                video_id,
                len(video_items),
            )
            video_embeddings: list[np.ndarray] = []
            saved_video_items: list[FrameItem] = []
            local_map_offset = 0
            for batch in chunked(video_items, int(PIPELINE_BATCH_SIZE)):
                if max_batches and global_batch_index >= int(max_batches):
                    break
                global_batch_index += 1
                batch_started = time.perf_counter()
                local_paths: list[Path] = []
                try:
                    download_started = time.perf_counter()
                    local_paths, downloaded_bytes = download_frame_batch(client, bucket_name, layout, batch)
                    download_ms = millis_since(download_started)

                    inference_started = time.perf_counter()
                    embeddings = encode_image_paths(local_paths)
                    inference_ms = millis_since(inference_started)

                    zilliz_started = time.perf_counter()
                    if WRITE_ZILLIZ_JSONL:
                        for local_idx, (item, vector) in enumerate(zip(batch, embeddings), start=1):
                            map_n = local_map_offset + local_idx
                            map_n_by_keyframe[item.keyframe_id] = map_n
                            zilliz_writer.write(
                                build_keyframe_zilliz_record(
                                    item=item,
                                    vector=vector,
                                    embedding_index_0=map_n - 1,
                                    map_n=map_n,
                                    dataset_db_id=dataset_db_id,
                                )
                            )
                    else:
                        for local_idx, item in enumerate(batch, start=1):
                            map_n_by_keyframe[item.keyframe_id] = local_map_offset + local_idx
                    zilliz_ms = millis_since(zilliz_started)

                    video_embeddings.append(embeddings)
                    saved_video_items.extend(batch)
                    local_map_offset += len(batch)
                    processed_frames += len(batch)
                    batch_ms = millis_since(batch_started)
                    percent = 100.0 * processed_frames / max(len(frames), 1)
                    speed = len(batch) / max(batch_ms / 1000, 1e-9)
                    metric = {
                        "batch_index": global_batch_index,
                        "video_id": video_id,
                        "frames": len(batch),
                        "processed_frames": processed_frames,
                        "total_frames": len(frames),
                        "percent": round(percent, 4),
                        "download_ms": download_ms,
                        "downloaded_bytes": downloaded_bytes,
                        "inference_ms": inference_ms,
                        "zilliz_write_ms": zilliz_ms,
                        "batch_ms": batch_ms,
                        "frames_per_second": round(speed, 4),
                    }
                    batch_metrics.append(metric)
                    logger.info(
                        "[batch %d/%d %.2f%%] video_id=%s frames=%d processed=%d/%d download_ms=%d inference_ms=%d zilliz_ms=%d total_ms=%d speed=%.2f fps",
                        global_batch_index,
                        total_batches,
                        percent,
                        video_id,
                        len(batch),
                        processed_frames,
                        len(frames),
                        download_ms,
                        inference_ms,
                        zilliz_ms,
                        batch_ms,
                        speed,
                    )
                except Exception as exc:
                    failed_frames += len(batch)
                    error = {
                        "mode": mode,
                        "run_id": run_id,
                        "video_id": video_id,
                        "batch_index": global_batch_index,
                        "frames": len(batch),
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                        "created_at": utc_now(),
                    }
                    errors.append(error)
                    logger.exception("batch failed video_id=%s batch_index=%d error=%s", video_id, global_batch_index, exc)
                    clear_gpu_cache()
                    if not CONTINUE_ON_BATCH_ERROR:
                        raise
                finally:
                    remove_local_downloads(local_paths)
            if video_embeddings:
                shape, map_rows = save_video_embeddings(layout, video_id, saved_video_items, video_embeddings)
                embedding_shapes[video_id] = shape
                processed_items_by_video[video_id] = list(saved_video_items)
                logger.info("saved video_id=%s embeddings_shape=%s map_rows=%d", video_id, shape, len(map_rows))
            if max_batches and global_batch_index >= int(max_batches):
                logger.info("stopping after max_batches=%d", int(max_batches))
                break
    finally:
        keyframe_parts = zilliz_writer.close()

    frames_by_video_processed = {
        video_id: items
        for video_id, items in processed_items_by_video.items()
        if (layout.feature_dir / f"{video_id}.npy").exists()
    }
    per_video_rows = build_video_summary_rows(frames_by_video_processed, embedding_shapes, layout)
    write_per_video_summary(layout, per_video_rows)
    write_csv(
        layout.artifacts_dir / "batch_metrics.csv",
        batch_metrics,
        [
            "batch_index",
            "video_id",
            "frames",
            "processed_frames",
            "total_frames",
            "percent",
            "download_ms",
            "downloaded_bytes",
            "inference_ms",
            "zilliz_write_ms",
            "batch_ms",
            "frames_per_second",
        ],
    )
    write_jsonl(layout.artifacts_dir / "errors.jsonl", errors)
    postgres_files = build_postgres_base_exports(
        layout,
        frames_by_video_processed,
        map_n_by_keyframe,
        embedding_shapes,
        bucket_name,
        run_id,
    )
    event_report = {"events": 0, "event_keyframes": 0, "event_zilliz_parts": [], "event_summary_rows": []}
    if BUILD_EVENTS:
        event_report = build_event_outputs(layout, frames_by_video_processed, logger)

    elapsed = time.perf_counter() - started
    summary = {
        "schema_version": "feature-extraction-v1",
        "task": TASK_NAME,
        "mode": mode,
        "run_id": run_id,
        "status": "success" if not errors else "partial",
        "created_at": utc_now(),
        "dataset": {
            "dataset_id": dataset_db_id,
            "dataset_code": DATASET_CODE,
            "dataset_name": DATASET_NAME,
            "dataset_version": DATASET_VERSION,
        },
        "input": {
            "bucket": bucket_name,
            "batches": selected,
            "sources": sources,
            "video_ids": VIDEO_IDS,
            "max_videos": max_videos,
            "max_frames": max_frames,
            "skip_frames": int(skip_frames),
            "loaded_frames_before_skip": loaded_frames_before_skip,
        },
        "model": model_metadata(),
        "parameters": run_parameters(),
        "collections": {
            "keyframe_embeddings": ZILLIZ_KEYFRAME_COLLECTION,
            "event_embeddings": ZILLIZ_EVENT_COLLECTION,
        },
        "totals": {
            "planned_frames": len(frames),
            "processed_frames": processed_frames,
            "failed_frames": failed_frames,
            "videos": len(frames_by_video_processed),
            "keyframe_vectors": processed_frames,
            "keyframe_zilliz_parts": len(keyframe_parts),
            "events": int(event_report.get("events", 0)),
            "event_keyframes": int(event_report.get("event_keyframes", 0)),
            "event_zilliz_parts": len(event_report.get("event_zilliz_parts", [])),
        },
        "timing": {
            "elapsed_seconds": round(elapsed, 3),
            "frames_per_second": round(processed_frames / max(elapsed, 1e-9), 4),
        },
        "artifacts": {
            "local_run_dir": str(layout.run_dir),
            "gcs_prefix": f"gs://{bucket_name}/{layout.gcs_prefix}" if upload_artifacts and UPLOAD_ARTIFACTS_TO_GCS else "",
            "per_video_summary": str(layout.run_dir / "per_video_summary.csv"),
            "postgres_files": {key: str(value) for key, value in postgres_files.items()},
            "keyframe_zilliz_parts": keyframe_parts,
            "event_zilliz_parts": event_report.get("event_zilliz_parts", []),
        },
    }
    write_json(layout.artifacts_dir / "summary.json", summary)
    write_run_report(layout, summary)

    uploaded_files = []
    if upload_artifacts and UPLOAD_ARTIFACTS_TO_GCS:
        logger.info("upload artifacts start prefix=gs://%s/%s", bucket_name, layout.gcs_prefix)
        uploaded_files = upload_tree(client, bucket_name, layout, logger)
        summary["artifacts"]["uploaded_files"] = uploaded_files
        write_json(layout.artifacts_dir / "summary.json", summary)
        upload_file(client, bucket_name, layout.artifacts_dir / "summary.json", layout.gcs_prefix + "artifacts/summary.json")
        logger.info("upload artifacts done files=%d", len(uploaded_files))

    logger.info(
        "run finished status=%s processed=%d failed=%d elapsed=%.2fs frames_per_second=%.2f",
        summary["status"],
        processed_frames,
        failed_frames,
        elapsed,
        processed_frames / max(elapsed, 1e-9),
    )
    print(json.dumps({k: summary[k] for k in ["run_id", "status", "totals", "timing", "artifacts"]}, ensure_ascii=False, indent=2))
    return summary


def run_dry_run() -> dict[str, Any]:
    """Run source discovery only."""
    return preview_plan(batches=BATCHES, max_videos=MAX_VIDEOS, max_frames=DRY_RUN_MAX_FRAMES)


def run_smoke_test() -> dict[str, Any]:
    """Run a tiny local extraction for environment and model validation."""
    return run_extraction(
        mode="smoke",
        batches=SMOKE_TEST_BATCHES,
        max_videos=1,
        max_frames=SMOKE_TEST_MAX_FRAMES,
        max_batches=1,
        skip_frames=0,
        upload_artifacts=SMOKE_TEST_UPLOAD_ARTIFACTS,
    )


def run_demo_one_batch() -> dict[str, Any]:
    """Run exactly one configured demo batch and optionally upload artifacts."""
    start = int(DEMO_BATCH_INDEX) * int(DEMO_BATCH_SIZE)
    original_video_ids = list(VIDEO_IDS)
    report = run_extraction(
        mode="demo",
        batches=DEMO_BATCHES,
        max_videos=None,
        max_frames=int(DEMO_BATCH_SIZE),
        max_batches=1,
        skip_frames=start,
        upload_artifacts=DEMO_UPLOAD_ARTIFACTS,
    )
    if start:
        print(f"Skipped {start} frame(s) before the demo batch.")
    assert original_video_ids == list(VIDEO_IDS)
    return report


def run_full() -> dict[str, Any]:
    """Run the full configured dataset after confirmation."""
    if CONFIRM_FULL_RUN != "RUN_FULL_DATASET":
        raise RuntimeError('Set CONFIRM_FULL_RUN = "RUN_FULL_DATASET" in Config before full run.')
    return run_extraction(
        mode="full",
        batches=FULL_BATCHES,
        max_videos=FULL_MAX_VIDEOS,
        max_frames=FULL_MAX_FRAMES,
        max_batches=None,
        skip_frames=0,
        upload_artifacts=True,
    )


# Dry Run

Chỉ list metadata frame và in sample. Không download ảnh, không load model, không upload artifact.

In [ ]:
dry_report = run_dry_run()


# Smoke Test

Chạy một lượng rất nhỏ frame để kiểm tra GCS download, model load, GPU inference, output CSV/NPY/JSONL.
Mặc định không upload artifact (`SMOKE_TEST_UPLOAD_ARTIFACTS = False`).

In [ ]:
smoke_report = run_smoke_test()


# Demo 1 Batch

Chạy đúng một outer batch theo `DEMO_BATCH_SIZE`, ghi đầy đủ artifact và upload lên GCS nếu
`DEMO_UPLOAD_ARTIFACTS = True`.

In [ ]:
demo_report = run_demo_one_batch()


# Full Run

Cell này có guard để tránh bấm nhầm `Run All`. Trước khi chạy toàn bộ, sửa `CONFIRM_FULL_RUN`
trong cell `Config` thành `"RUN_FULL_DATASET"`.

In [ ]:
if CONFIRM_FULL_RUN == "RUN_FULL_DATASET":
    full_report = run_full()
else:
    print('Full run is locked. Set CONFIRM_FULL_RUN = "RUN_FULL_DATASET" in Config to run all data.')


# Inspect Latest Output

Cell này liệt kê các run gần nhất trong `/kaggle/working/feature_extraction_runs` và các file
quan trọng để bạn kiểm tra tốc độ, lỗi, summary, payload Supabase và Zilliz.

In [ ]:
run_root = Path(LOCAL_RUN_ROOT)
latest_runs = sorted(
    [path for path in run_root.glob("*") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)[:5]

if not latest_runs:
    print("No local runs found yet.")

for run_dir in latest_runs:
    print("\nRUN:", run_dir)
    for rel in [
        "artifacts/summary.json",
        "artifacts/report.md",
        "artifacts/batch_metrics.csv",
        "artifacts/errors.jsonl",
        "per_video_summary.csv",
        f"features/{FEATURE_DIR_NAME}",
        "features/map-keyframes",
        "features/events",
        "features/map-event",
        "postgres",
        "zilliz",
        "run.log",
    ]:
        path = run_dir / rel
        if path.exists():
            if path.is_dir():
                count = sum(1 for _ in path.rglob("*") if _.is_file())
                print("  ", rel, f"({count} files)")
            else:
                print("  ", rel, path.stat().st_size, "bytes")
